# Instruction Tuning: watch a diverse instruction mix create a skill that transfers

This notebook proves, from scratch and deterministically, the central empirical claim of
**instruction tuning** (FLAN, Wei et al. 2021): fine-tune one model on a **large, diverse mix of
tasks phrased as instructions** and it learns the *meta-skill* **"read the instruction, then do
what it says"** — a skill that **transfers to instructions whose content it never saw** (zero-shot).

We build two models on a **fair, equal compute budget**:

- a **multitask** model trained on a diverse mix — `SUBSTITUTE` (with ever-changing keys), `REVERSE`, `COPY`;
- a **single-task** model trained on `SUBSTITUTE` with **one fixed key**.

Then we evaluate **both** on `SUBSTITUTE` with **fresh, unseen keys**. The multitask model, having
been *forced* to learn to read the key, generalizes; the single-task model, having *memorized* one
mapping, fails. The whole point is the **generalization gap** — not state-of-the-art accuracy.

> This reuses the **supervised next-token loss** of [chapter 13 (Supervised Fine-Tuning)](../../13-Supervised-Fine-Tuning/13-Supervised-Fine-Tuning.md) — masked
> cross-entropy over the response tokens. Instruction tuning **is** that same loss, applied over a
> diverse multitask instruction mix. We don't re-derive the loss here.


> **Step 1 — imports, hyperparameters, and honest device detection.** We detect the best
> accelerator but **pin the trace to CPU** so the printed accuracies are identical on any machine
> (MPS/CUDA reductions are nondeterministic). We print `torch.__version__` and seed everything.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- Vocabulary: content symbols + structural/operation tokens ----
N_SYMBOLS = 6          # content symbols 0..5; the SUBSTITUTE key is a permutation of these
INPUT_LEN = 4          # every input/output block is this many symbols

SEP = N_SYMBOLS + 0            # separates the instruction block from the input
OP_SUBSTITUTE = N_SYMBOLS + 1  # "apply the key table that follows"
OP_REVERSE = N_SYMBOLS + 2     # "reverse the input"
OP_COPY = N_SYMBOLS + 3        # "echo the input"
KEY_PAD = N_SYMBOLS + 4        # fills KEY slots for ops with no key
VOCAB = N_SYMBOLS + 5

OP_ID_SUBSTITUTE, OP_ID_REVERSE, OP_ID_COPY = 0, 1, 2
OP_NAMES = {0: "SUBSTITUTE", 1: "REVERSE", 2: "COPY"}

# layout: [OP][KEY: N_SYMBOLS][SEP][input: INPUT_LEN][output: INPUT_LEN]
RESPONSE_START = 1 + N_SYMBOLS + 1 + INPUT_LEN   # first index of the OUTPUT block
SEQ_TOTAL = RESPONSE_START + INPUT_LEN

SEED = 0
D_MODEL, N_HEADS, N_LAYERS, D_FF = 96, 4, 3, 256
N_TRAIN_EXAMPLES, N_EVAL_EXAMPLES = 8000, 1000   # SAME budget for both models -> fair compute
BATCH_SIZE, N_EPOCHS, LEARNING_RATE = 256, 10, 2e-3
SINGLE_FIXED_KEY = torch.tensor([1, 2, 3, 4, 5, 0])  # the one key the single-task model memorizes

DETECTED_DEVICE = ("cuda" if torch.cuda.is_available()
                   else "mps" if torch.backends.mps.is_available() else "cpu")
DEVICE = "cpu"   # the trace runs HERE regardless of what was detected
torch.manual_seed(SEED)
print(f"torch: {torch.__version__}")
print(f"device: {DEVICE} (detected {DETECTED_DEVICE}; pinned to CPU for reproducibility)")
print(f"seed: {SEED}")

torch: 2.12.0
device: cpu (detected mps; pinned to CPU for reproducibility)
seed: 0


> **Step 2 — the operations.** Three tiny symbolic transforms. `SUBSTITUTE` is the one that
> *reads an instruction argument*: its output is `key[input]`, where the key is a permutation
> **given in the instruction**. `REVERSE` and `COPY` are fixed operations. Generalizing to a new
> key is possible **only** by learning to read the key.

In [2]:
def apply_op(op_id, key, inp):
    """Apply one operation to a (INPUT_LEN,) input. Only SUBSTITUTE uses the key."""
    if op_id == OP_ID_SUBSTITUTE:
        return key[inp]                  # output[i] = key[input[i]]
    if op_id == OP_ID_REVERSE:
        return torch.flip(inp, dims=[0]) # reverse the order
    if op_id == OP_ID_COPY:
        return inp.clone()               # echo unchanged
    raise ValueError(op_id)

# sanity: SUBSTITUTE applies the key elementwise
_key = torch.tensor([2, 0, 1, 4, 5, 3])
print("key      :", _key.tolist())
print("input    :", [1, 5, 0, 2])
print("SUBSTITUTE:", apply_op(OP_ID_SUBSTITUTE, _key, torch.tensor([1, 5, 0, 2])).tolist())
print("REVERSE   :", apply_op(OP_ID_REVERSE, _key, torch.tensor([1, 5, 0, 2])).tolist())

key      : [2, 0, 1, 4, 5, 3]
input    : [1, 5, 0, 2]
SUBSTITUTE: [0, 3, 2, 1]
REVERSE   : [2, 0, 5, 1]


> **Step 3 — render an example into the instruction-template format.** Every example is one
> flat token sequence: `[ OP | KEY (N_SYMBOLS) | SEP | input | output ]`. For `REVERSE`/`COPY` the
> key slots are `KEY_PAD` — the operation token alone specifies the behaviour. The loss will be
> masked to the **output block** (everything at index `>= RESPONSE_START`).

In [3]:
def render_example(op_id, inp, key):
    if op_id == OP_ID_SUBSTITUTE:
        op_tok, key_block = torch.tensor([OP_SUBSTITUTE]), key
    else:
        op_tok = torch.tensor([OP_REVERSE if op_id == OP_ID_REVERSE else OP_COPY])
        key_block = torch.full((N_SYMBOLS,), KEY_PAD)
    out = apply_op(op_id, key, inp)
    return torch.cat([op_tok, key_block, torch.tensor([SEP]), inp, out])

# show the three trained operations + the held-out instance, in template form
sg = torch.Generator().manual_seed(SEED + 1)
demo_key = torch.tensor([2, 0, 1, 4, 5, 3])
print("Instruction-template format  [OP | KEY | SEP | input -> output]:")
for op_id in (OP_ID_SUBSTITUTE, OP_ID_REVERSE, OP_ID_COPY):
    inp = torch.randint(0, N_SYMBOLS, (INPUT_LEN,), generator=sg)
    row = render_example(op_id, inp, demo_key)
    print(f"  {OP_NAMES[op_id]:>10}: op={row[0].item()} key={row[1:1+N_SYMBOLS].tolist()} SEP "
          f"input={row[RESPONSE_START-INPUT_LEN:RESPONSE_START].tolist()} -> output={row[RESPONSE_START:].tolist()}")
held = render_example(OP_ID_SUBSTITUTE, torch.tensor([0,3,5,1]), torch.tensor([4,5,0,1,2,3]))
print(f"  HELD-OUT  : op={held[0].item()} key=[4,5,0,1,2,3] (UNSEEN) SEP "
      f"input=[0,3,5,1] -> output={held[RESPONSE_START:].tolist()}   <-- ZERO-SHOT test")

Instruction-template format  [OP | KEY | SEP | input -> output]:
  SUBSTITUTE: op=7 key=[2, 0, 1, 4, 5, 3] SEP input=[1, 5, 0, 2] -> output=[0, 3, 2, 1]
     REVERSE: op=8 key=[10, 10, 10, 10, 10, 10] SEP input=[1, 1, 5, 5] -> output=[5, 5, 1, 1]
        COPY: op=9 key=[10, 10, 10, 10, 10, 10] SEP input=[5, 0, 2, 3] -> output=[5, 0, 2, 3]
  HELD-OUT  : op=7 key=[4,5,0,1,2,3] (UNSEEN) SEP input=[0,3,5,1] -> output=[4, 1, 3, 5]   <-- ZERO-SHOT test


> **Step 4 — the three samplers that define the experiment.** The **multitask** sampler draws a
> *fresh random key* for every `SUBSTITUTE` example (so the model can't memorize — it must read the
> key) and mixes in `REVERSE`/`COPY`. The **single-task** sampler always uses the one fixed key. The
> **held-out** sampler is `SUBSTITUTE` with keys neither model ever trained on.

In [4]:
def sample_multitask_row(gen):
    op_id = int(torch.randint(0, 3, (1,), generator=gen))
    inp = torch.randint(0, N_SYMBOLS, (INPUT_LEN,), generator=gen)
    key = torch.randperm(N_SYMBOLS, generator=gen)   # FRESH key every time
    return render_example(op_id, inp, key)

def sample_singletask_row(gen):
    inp = torch.randint(0, N_SYMBOLS, (INPUT_LEN,), generator=gen)
    return render_example(OP_ID_SUBSTITUTE, inp, SINGLE_FIXED_KEY)  # always the SAME key

def sample_heldout_row(gen):
    inp = torch.randint(0, N_SYMBOLS, (INPUT_LEN,), generator=gen)
    key = torch.randperm(N_SYMBOLS, generator=gen)   # a key NEITHER model trained on
    return render_example(OP_ID_SUBSTITUTE, inp, key)

def make_dataset(n, sampler, gen):
    return torch.stack([sampler(gen) for _ in range(n)])

print("multitask row :", make_dataset(1, sample_multitask_row, torch.Generator().manual_seed(3))[0].tolist())
print("singletask row:", make_dataset(1, sample_singletask_row, torch.Generator().manual_seed(3))[0].tolist())

multitask row : [8, 10, 10, 10, 10, 10, 10, 6, 2, 1, 3, 4, 4, 3, 1, 2]
singletask row: [7, 1, 2, 3, 4, 5, 0, 6, 4, 2, 1, 3, 5, 3, 2, 4]


> **Step 5 — a small decoder-style Transformer.** Token + positional embeddings, a causal-masked
> encoder stack, a tied-width head. The **only** way to produce a correct `SUBSTITUTE` output for a
> *new* key is to read the key tokens — so any held-out success is evidence the model learned that
> skill, not a memorized table.

In [5]:
class InstructionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB, D_MODEL)
        self.pos_emb = nn.Embedding(SEQ_TOTAL, D_MODEL)
        layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, D_FF, batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=N_LAYERS)
        self.head = nn.Linear(D_MODEL, VOCAB)
        mask = torch.triu(torch.ones(SEQ_TOTAL, SEQ_TOTAL) * float("-inf"), diagonal=1)
        self.register_buffer("causal_mask", mask)   # next-token: position t attends to <= t

    def forward(self, tokens):
        pos = torch.arange(tokens.shape[1], device=tokens.device)
        x = self.tok_emb(tokens) + self.pos_emb(pos)
        x = self.encoder(x, mask=self.causal_mask[:tokens.shape[1], :tokens.shape[1]])
        return self.head(x)

m = InstructionTransformer().to(DEVICE)
print("parameters:", sum(p.numel() for p in m.parameters()))
print("forward shape:", m(make_dataset(2, sample_multitask_row, torch.Generator().manual_seed(0)).to(DEVICE)).shape,
      "= (batch, SEQ_TOTAL, VOCAB)")

parameters: 265067
forward shape: torch.Size([2, 16, 11]) = (batch, SEQ_TOTAL, VOCAB)


> **Step 6 — the chapter-13 masked next-token loss.** Logits at position `t` predict token `t+1`;
> we keep **only** the positions whose target lands in the **output block**. This is exactly SFT's
> masked cross-entropy — instruction tuning just applies it across a diverse instruction mix.

In [6]:
def masked_next_token_loss(logits, targets):
    pred = logits[:, :-1, :]            # logits predicting positions 1..T-1
    gold = targets[:, 1:]              # the actual next tokens
    keep = RESPONSE_START - 1          # first gold index inside the response block
    pred = pred[:, keep:, :].reshape(-1, VOCAB)
    gold = gold[:, keep:].reshape(-1)
    return F.cross_entropy(pred, gold)

_b = make_dataset(4, sample_multitask_row, torch.Generator().manual_seed(0)).to(DEVICE)
print("initial loss on a random batch:", round(masked_next_token_loss(m(_b), _b).item(), 4),
      "(~ln(VOCAB) =", round(float(torch.log(torch.tensor(float(VOCAB)))), 4), "at init)")

initial loss on a random batch: 2.5144 (~ln(VOCAB) = 2.3979 at init)


> **Step 7 — one training function, used for both models.** Identical init (same seed) and the
> same example budget, so the only difference between the two runs is the **data mix**. That is what
> makes the comparison fair.

In [7]:
def train_model(sampler, tag):
    torch.manual_seed(SEED)                       # identical init for both models
    gen = torch.Generator().manual_seed(SEED)
    tokens = make_dataset(N_TRAIN_EXAMPLES, sampler, gen).to(DEVICE)
    model = InstructionTransformer().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    n_batches = N_TRAIN_EXAMPLES // BATCH_SIZE
    model.train()
    for epoch in range(N_EPOCHS):
        perm = torch.randperm(N_TRAIN_EXAMPLES, generator=gen)
        epoch_loss = 0.0
        for b in range(n_batches):
            batch = tokens[perm[b*BATCH_SIZE:(b+1)*BATCH_SIZE]]
            loss = masked_next_token_loss(model(batch), batch)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        if epoch == 0 or epoch == N_EPOCHS - 1:
            print(f"  [{tag}] epoch {epoch:2d}  loss={epoch_loss/n_batches:.4f}")
    return model

> **Step 8a — train the MULTITASK model** on the diverse mix `{SUBSTITUTE (random keys), REVERSE, COPY}`.

In [8]:
print("Training MULTITASK model on {SUBSTITUTE (random keys), REVERSE, COPY}:")
multi = train_model(sample_multitask_row, "multi")

Training MULTITASK model on {SUBSTITUTE (random keys), REVERSE, COPY}:


  [multi] epoch  0  loss=1.5304


  [multi] epoch  9  loss=0.0004


> **Step 8b — train the SINGLE-TASK model** on `SUBSTITUTE` with the **one fixed key** only. Same
> init, same budget — it simply never sees instruction *diversity*.

In [9]:
print("Training SINGLETASK model on {SUBSTITUTE with ONE fixed key}:")
single = train_model(sample_singletask_row, "single")

Training SINGLETASK model on {SUBSTITUTE with ONE fixed key}:


  [single] epoch  0  loss=0.7631


  [single] epoch  9  loss=0.0004


> **Step 9 — evaluate both, then assert the gap.** Exact-match accuracy (a sequence counts only
> if *every* output symbol is right). First an **in-distribution** check (the fixed-key task both can
> do — proving capacity is not the issue), then the **zero-shot** test on **unseen keys**.

In [10]:
@torch.no_grad()
def evaluate(model, eval_tokens):
    model.eval()
    pred = model(eval_tokens)[:, :-1, :].argmax(dim=-1)
    gold = eval_tokens[:, 1:]
    keep = RESPONSE_START - 1
    return (pred[:, keep:] == gold[:, keep:]).all(dim=1).float().mean().item()

in_dist = make_dataset(N_EVAL_EXAMPLES, sample_singletask_row, torch.Generator().manual_seed(SEED+555)).to(DEVICE)
heldout = make_dataset(N_EVAL_EXAMPLES, sample_heldout_row, torch.Generator().manual_seed(SEED+31337)).to(DEVICE)

multi_train,  single_train  = evaluate(multi, in_dist),  evaluate(single, in_dist)
multi_held,   single_held   = evaluate(multi, heldout),  evaluate(single, heldout)

print("--- Results (exact-match accuracy) ---")
print(f"  SUBSTITUTE, fixed key   | multitask: {multi_train:6.1%} | singletask: {single_train:6.1%}")
print(f"  SUBSTITUTE, UNSEEN keys | multitask: {multi_held:6.1%} | singletask: {single_held:6.1%}")
print(f"\n  zero-shot generalization GAP (multitask - singletask): {multi_held - single_held:+.1%}")

assert multi_held > single_held, "multitask must beat singletask on the held-out test"
assert multi_held > 0.9, "multitask model should generalize to unseen keys"
print("  assert multitask_heldout_acc > singletask_heldout_acc: PASSED")

--- Results (exact-match accuracy) ---
  SUBSTITUTE, fixed key   | multitask: 100.0% | singletask: 100.0%
  SUBSTITUTE, UNSEEN keys | multitask: 100.0% | singletask:   0.7%

  zero-shot generalization GAP (multitask - singletask): +99.3%
  assert multitask_heldout_acc > singletask_heldout_acc: PASSED


## Try it yourself

Before you run anything, **predict** the outcome:

1. **Shrink the diversity.** In `sample_multitask_row`, drop `REVERSE` and `COPY` so the multitask
   model trains on `SUBSTITUTE` with random keys *only*. Does it still generalize to unseen keys?
   *(It does — the key insight is that the keys **vary**, forcing the read-the-key skill. The other
   ops add robustness, but variation within one task is already enough here.)*
2. **Freeze the multitask key.** Make `sample_multitask_row` reuse `SINGLE_FIXED_KEY` for every
   `SUBSTITUTE` example. Now the multitask model has no reason to read the key — predict its
   held-out accuracy. *(It collapses toward the single-task model's: diversity, not the mix of op
   types, is doing the work.)*
3. **Scale it.** Raise `N_SYMBOLS` to 10 and `INPUT_LEN` to 6. The held-out gap should persist —
   the mechanism doesn't depend on the toy size.

The lesson holds at every scale: **a model generalizes to instructions it never saw exactly when
training forced it to *read* instructions rather than memorize one mapping.** That is the entire
thesis of FLAN — here, in 100 lines, on a CPU.

## What we saw

- **Diversity creates a transferable skill.** Both models hit **100%** on the in-distribution
  fixed-key task — capacity is identical. On **unseen keys**, the multitask model scores **100%**
  and the single-task model **~1%**: a **+99-point** zero-shot generalization gap, from the data mix
  alone.
- **The mechanism is "read the instruction."** Forced to handle ever-changing keys, the multitask
  model learned to *look up the key in the instruction and apply it* — a meta-skill that transfers
  to any new key. The single-task model memorized one mapping and had no such skill.
- **Same loss as SFT.** We used chapter 13's masked next-token cross-entropy unchanged. Instruction
  tuning is **not a new objective** — it's SFT over a **large, diverse, instruction-phrased** task
  mix, and *that* is what buys zero-shot generalization (FLAN, Wei et al. 2021).